In [11]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [12]:
model = pe.ConcreteModel()

### Sets

In [13]:
plane_types = ['A', 'B', 'C']
number_planes = [1, 2, 3, 4, 5]

model.plane_types = pe.Set(initialize=plane_types)
model.number_planes = pe.Set(initialize=number_planes)

### Parameters

In [14]:
cost_dict = {
    ("A", 1): 11, ("A", 2): 20, ("A", 3): 30, ("A", 4): 40, ("A", 5): 50,
    ("B", 1): 9, ("B", 2): 17, ("B", 3): 24, ("B", 4): 34, ("B", 5): 45,
    ("C", 1): 8, ("C", 2): 15, ("C", 3): 21, ("C", 4): 26, ("C", 5): 31
}
model.cost = pe.Param(model.plane_types, model.number_planes, initialize=cost_dict)
model.fixed_cost = pe.Param(initialize=6)

In [15]:
capacity_dict = {
    "A": 80, "B": 68, "C": 55
}
model.capacity = pe.Param(model.plane_types, initialize=capacity_dict)
model.number_passengers = pe.Param(initialize=372)

### Variables

In [16]:
model.x = pe.Var(model.plane_types, model.number_planes, domain=pe.Binary)

### Funcion objetivo

Minimizar el coste de los aviones para transportar a todos los pasajeros

In [17]:
def obj_rule(model):
    return sum(model.cost[p, n] * model.x[p, n] for p in model.plane_types for n in model.number_planes) + model.fixed_cost * sum([1 for i in model.plane_types for j in model.number_planes if model.x[i, j].value == 1])

### Restricciones

In [18]:
def pasengers(model):
    return sum(model.capacity[p] * model.x[p, n] for p in model.plane_types for n in model.number_planes) >= model.number_passengers

In [20]:
def max_type_selection(model, p):
    return sum(model.x[p, n] for n in model.number_planes) <= 1

model.max_selection = pe.Constraint(model.plane_types, rule=max_type_selection)

(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().


## Resolución con gurobi

In [21]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=True)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 3 rows, 15 columns and 15 nonzeros (Min)
Model fingerprint: 0xe569cd0a
Model has 0 linear objective coefficients
Variable types: 0 continuous, 15 integer (15 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 22 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%


In [25]:
plane_cost = sum(
    pe.value(model.cost[p, n] * model.x[p, n]) for p in model.plane_types for n in model.number_planes
)

type_cost = model.fixed_cost * sum(
    1 for i in model.plane_types for j in model.number_planes if pe.value(model.x[i, j]) == 1
)

print(f"Coste de combustible: {plane_cost:.2f} €")
print(f"Coste por cambios:   {type_cost:.2f} €")
print(f"COSTE TOTAL MÍNIMO:  {plane_cost + type_cost:.2f} €")

Coste de combustible: 0.00 €
Coste por cambios:   0.00 €
COSTE TOTAL MÍNIMO:  0.00 €
